# Práctica de Laboratorio 3: Políticas Basadas en Gradiente y Métodos Actor-Critic

**Curso:** DS5345 Aprendizaje por Refuerzo
**Duración:** 100 minutos  
**Modalidad:** Estrictamente individual  
**Entrega:** Subir este archivo `.ipynb` ejecutado con código completo, gráficas generadas y respuestas a las preguntas de análisis a Canvas.

---

### Instrucciones para el Estudiante:

- Esta es la **plantilla de trabajo** oficial para la práctica.
- Deberás completar todos los bloques de código señalados con **`# TODO:`** y **`# === TU CÓDIGO AQUÍ ===`**.
- **Almacenamiento de Métricas:** Es indispensable que tus funciones almacenen y retornen las variables indicadas (`rewards_history`, `ep_rewards_history`, `clip_fractions_history`), ya que los bloques de graficación provistos dependen estrictamente de ellas.
- **Preguntas de Análisis:** Responde de forma rigurosa y con fundamentos matemáticos a las preguntas 1.4, 2.4 y 2.5 en las celdas Markdown designadas (`[RESPUESTA DEL ESTUDIANTE]`).


In [ ]:
# =====================================================================
# CONFIGURACIÓN INICIAL Y LIBRERÍAS
# =====================================================================
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.distributions import Categorical, Normal
import matplotlib.pyplot as plt

# Fijar semillas para reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch configurado en dispositivo: {device}")

PyTorch configurado en dispositivo: cpu


## 1. Reducción de Varianza en Ambientes Discretos (8 puntos)

### Objetivo:

Evaluar y comparar empíricamente la estabilidad y la varianza en el aprendizaje de un método Monte Carlo (**REINFORCE con Baseline**) frente a un método de Diferencia Temporal (TD) de un paso (**A2C**). Para ello, utilizaremos el entorno **`CartPole-v1`** de Gymnasium.


In [ ]:
# =====================================================================
# ARQUITECTURAS DE RED PARA AMBIENTES DISCRETOS (CartPole-v1)
# =====================================================================


class DiscretePolicyNetwork(nn.Module):
    """Actor independiente para REINFORCE con Baseline (Apartado 1.1)."""

    def __init__(self, state_dim, action_dim, hidden=128):
        super(DiscretePolicyNetwork, self).__init__()
        # TODO: Define la red neuronal del Actor (Ejm. Linear(state_dim, hidden) -> ReLU -> Linear(hidden, action_dim))
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, action_dim),
        )

    def forward(self, x):
        # TODO: Retorna las probabilidades sobre las acciones aplicando softmax en la última dimensión (dim=-1)
        return Categorical(logits=self.net(x))


class ValueNetwork(nn.Module):
    """Crítico independiente V(s) para función de baseline en REINFORCE (Apartado 1.1)."""

    def __init__(self, state_dim, hidden=128):
        super(ValueNetwork, self).__init__()
        # TODO: Define la red neuronal del Crítico para estimar V(s) (Ejm. Linear(state_dim, hidden) -> ReLU -> Linear(hidden, 1))
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        # TODO: Retorna la estimación escalar del valor V(s)
        return self.net(x)


class DiscreteActorCriticShared(nn.Module):
    """
    Arquitectura con tronco compartido para A2C Discreto (Apartado 1.2 del PDF).
    El tronco extrae características comunes y dos cabezas separadas
    estiman la distribución de política Categorical y el valor escalar V(s).
    """

    def __init__(self, state_dim, action_dim, hidden=128):
        super(DiscreteActorCriticShared, self).__init__()
        # TODO: Define el tronco compartido (shared trunk) con al menos una capa oculta y activación ReLU
        self.tronco = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        # TODO: Define la cabeza del Actor (salida: action_dim) y la cabeza del Crítico (salida: 1)
        self.actor = nn.Linear(hidden, action_dim)
        self.critic = nn.Linear(hidden, 1)

    def forward(self, x):
        # TODO: Pasa la entrada x por el tronco compartido
        features = self.tronco(x)

        # TODO: Calcula las probabilidades de acción (softmax) y el valor V(s)
        dist = Categorical(probs=F.softmax(self.actor(features), dim=-1))
        value = self.critic(features)

        # Retorna: dist (objeto Categorical(probs)), value (tensor de valor de estado)
        return dist, value

In [ ]:
def train_reinforce_baseline(env_name="CartPole-v1", episodes=600, lr=3e-4, gamma=0.99):
    """
    Apartado 1.1: REINFORCE con Baseline independiente.

    Requisitos:
    1. Recolectar trayectorias completas de cada episodio (Monte Carlo).
    2. Calcular los retornos acumulados descontados G_t hacia atrás: G_t = r_{t+1} + gamma * G_{t+1}.
    3. Emplear V(s_t) como baseline para calcular la ventaja: A_t = G_t - V(s_t).
    4. Calcular la pérdida del actor: -log_prob * advantage.
    5. Actualizar la red de valor minimizando el error cuadrático medio (MSE) respecto a G_t.
    6. Almacenar la recompensa total de cada episodio en `rewards_history`.
    """
    torch.manual_seed(42)
    np.random.seed(42)
    env = gym.make(env_name)
    env.reset(seed=42)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    actor = DiscretePolicyNetwork(state_dim, action_dim).to(device)
    critic = ValueNetwork(state_dim).to(device)

    optimizer_actor = optim.Adam(actor.parameters(), lr=lr)
    optimizer_critic = optim.Adam(critic.parameters(), lr=lr)

    # IMPORTANTE: Almacena aquí la suma de recompensas de cada episodio para que
    # el bloque de graficación provisto funcione automáticamente.
    rewards_history = []

    for episode in range(episodes):
        state, _ = env.reset()
        log_probs = []
        values = []
        rewards = []

        done = False
        while not done:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)

            # TODO: Evalúa el estado en el actor y el crítico
            # Muestrea una acción discreta usando la distribución Categorical
            # Calcula log_prob y guarda log_prob, valor y reward en sus listas

            dist = actor(state_tensor)
            val = critic(state_tensor)

            action = dist.sample()
            log_prob = dist.log_prob(action)
            val = val.squeeze(0)

            next_state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated

            log_probs.append(log_prob)
            values.append(val)
            rewards.append(reward)

            state = next_state

        # TODO 1.1.A: Guarda la suma de recompensas del episodio actual en rewards_history
        rewards_history.append(sum(rewards))

        # TODO 1.1.B: Calcular retornos descontados G_t hacia atrás (Monte Carlo)
        # G_t = r_{t+1} + gamma * G_{t+1}

        returns = []
        G = 0.0

        for r in reversed(rewards):
            G = r + gamma * G
            returns.insert(0, G)

        returns = torch.tensor(returns, dtype=torch.float32, device=device)

        # TODO 1.1.C: Calcular la pérdida del actor y del crítico empleando el baseline V(s_t)
        # Ventaja: Advantage = G_t - V(s_t)
        # Pérdida del Actor: sum( -log_prob * Advantage )
        # Pérdida del Crítico: MSE entre V(s_t) y G_t

        log_probs = torch.cat(log_probs)
        values = torch.cat(values)
        advantages = returns - values.detach()

        actor_loss = -(log_probs * advantages).sum()
        critic_loss = (values - returns).pow(2).mean()

        # TODO 1.1.D: Actualizar parámetros de ambas redes con sus respectivos optimizadores
        # (optimizer_actor y optimizer_critic)

        optimizer_actor.zero_grad()
        actor_loss.backward()
        optimizer_actor.step()

        optimizer_critic.zero_grad()
        critic_loss.backward()
        optimizer_critic.step()

    env.close()
    return rewards_history

In [ ]:
def train_a2c_discrete(env_name="CartPole-v1", episodes=600, lr=3e-4, gamma=0.99):
    """
    Apartado 1.2: Advantage Actor-Critic (A2C) Discreto con tronco compartido.

    Requisitos:
    1. Utilizar la arquitectura con tronco compartido `DiscreteActorCriticShared`.
    2. En cada paso de tiempo, calcular la ventaja basada en el error TD de un paso:
           A_t = r_{t+1} + gamma * V(s_{t+1}) - V(s_t)
    3. Calcular la pérdida total: Actor Loss (-log_prob * A_t.detach()) + Critic Loss (MSE).
    4. Realizar la actualización de parámetros on-policy tras cada paso.
    5. Almacenar la recompensa total de cada episodio en `rewards_history`.
    """
    torch.manual_seed(42)
    np.random.seed(42)
    env = gym.make(env_name)
    env.reset(seed=42)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.n

    model = DiscreteActorCriticShared(state_dim, action_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # IMPORTANTE: Almacena aquí la suma de recompensas de cada episodio
    rewards_history = []

    for episode in range(episodes):
        state, _ = env.reset()
        episode_reward = 0

        done = False
        while not done:
            state_tensor = torch.as_tensor(
                state, dtype=torch.float32, device=device
            ).unsqueeze(0)

            # TODO: Obtener distribución Categorical y valor V(s_t) del estado actual desde el modelo
            # Muestrear una acción
            # === TU CÓDIGO AQUÍ ===
            dist, val = model(state_tensor)
            action = dist.sample()

            next_state, reward, terminated, truncated, _ = env.step(action.item())
            done = terminated or truncated

            # TODO 1.2.A: Obtener V(s_{t+1}) para el siguiente estado (bootstrap)
            # Recordar: si el episodio terminó (done=True), val_next = torch.tensor([[0.0]])
            # === TU CÓDIGO AQUÍ ===
            with torch.no_grad():
                next_state_tensor = torch.as_tensor(
                    next_state, dtype=torch.float32, device=device
                ).unsqueeze(0)
                _, val_next = model(next_state_tensor)
                if done:
                    val_next = torch.zeros_like(val_next)

            # TODO 1.2.B: Calcular el TD Target y la Ventaja TD de 1 paso:
            # td_target = reward + gamma * val_next
            # advantage = td_target - val
            # === TU CÓDIGO AQUÍ ===
            td_target = reward + gamma * val_next
            advantage = td_target - val

            # TODO 1.2.C: Calcular pérdidas y optimizar on-policy
            # Actor Loss: -dist.log_prob(action) * advantage.detach()
            # Critic Loss: F.mse_loss(val, td_target)
            # Loss = Actor Loss + 0.5 * Critic Loss
            # Actualizar con optimizer.zero_grad(), loss.backward(), optimizer.step()
            # === TU CÓDIGO AQUÍ ===
            actor_loss = -(dist.log_prob(action) * advantage.detach()).mean()
            critic_loss = F.mse_loss(val, td_target)
            loss = actor_loss + 0.5 * critic_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            state = next_state
            episode_reward += reward

        # TODO 1.2.D: Guardar la recompensa total del episodio en rewards_history
        # === TU CÓDIGO AQUÍ ===
        rewards_history.append(episode_reward)

    env.close()
    return rewards_history

In [ ]:
# =====================================================================
# BLOQUE DE EJECUCIÓN Y VISUALIZACIÓN DEL EJERCICIO 1 (PROVISTO)
# =====================================================================


def plot_moving_average(data, window_size=20):
    """Calcula la media móvil para suavizar las curvas de aprendizaje."""
    if len(data) < window_size:
        return np.array(data)
    return np.convolve(data, np.ones(window_size) / window_size, mode="valid")


print("===============================================================")
print(" ENTRENAMIENTO EJERCICIO 1: CartPole-v1 (600 Episodios)")
print("===============================================================")

episodes_ex1 = 600

print(f"Entrenando REINFORCE con Baseline ({episodes_ex1} episodios)...")
rewards_reinforce = train_reinforce_baseline(
    env_name="CartPole-v1", episodes=episodes_ex1
)

print(f"Entrenando A2C Discreto con Tronco Compartido ({episodes_ex1} episodios)...")
rewards_a2c_disc = train_a2c_discrete(env_name="CartPole-v1", episodes=episodes_ex1)

# Visualización comparativa (Apartado 1.3)
plt.figure(figsize=(12, 6))

# Curvas de datos crudos (tenues)
plt.plot(rewards_reinforce, alpha=0.25, color="blue", label="REINFORCE (Datos crudos)")
plt.plot(rewards_a2c_disc, alpha=0.25, color="orange", label="A2C (Datos crudos)")

# Curvas suavizadas con media móvil de 20 episodios
ma_reinforce = plot_moving_average(rewards_reinforce, 20)
ma_a2c = plot_moving_average(rewards_a2c_disc, 20)
plt.plot(
    range(19, 19 + len(ma_reinforce)),
    ma_reinforce,
    color="blue",
    linewidth=2.2,
    label="REINFORCE (Media Móvil 20)",
)
plt.plot(
    range(19, 19 + len(ma_a2c)),
    ma_a2c,
    color="orange",
    linewidth=2.2,
    label="A2C (Media Móvil 20)",
)

plt.title(
    "Ejercicio 1.3: Comparación de REINFORCE con Baseline vs A2C en CartPole-v1",
    fontsize=14,
    fontweight="bold",
)
plt.xlabel("Episodios", fontsize=12)
plt.ylabel("Recompensa Acumulada", fontsize=12)
plt.axhline(
    y=475, color="green", linestyle="--", label="Umbral de Dominio CartPole-v1 (475)"
)
plt.legend(loc="upper left")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### 1.4. Pregunta de Análisis: Varianza y Rol del Crítico (1.5 puntos)

_Explique con fundamentos matemáticos analizados en clase, por qué la curva de REINFORCE presenta fluctuaciones significativamente mayores (varianza) en comparación con A2C y de qué manera el Crítico actúa como un reductor de varianza efectivo._

---

**Tu respuesta debe estructurarse considerando:**

1. **Retorno Monte Carlo:** Escribe la expresión de $\nabla_\theta J(\theta)$ en REINFORCE y explica por qué la suma completa de recompensas futuras $G_t = \sum_{k=0}^{T-t-1} \gamma^k r_{t+k+1}$ acumula una alta varianza a lo largo de la trayectoria.
2. **Propiedad del Baseline:** Demuestra o explica formalmente por qué restar un baseline $b(s_t) = V_\phi(s_t)$ no introduce sesgo en la estimación del gradiente (propiedad $\mathbb{E}[\nabla_\theta \log \pi_\theta \cdot b] = 0$).
3. **Bootstrapping en A2C:** Explica cómo el error TD de un paso $A_t = r_{t+1} + \gamma V_\phi(s_{t+1}) - V_\phi(s_t)$ sustituye la cola estocástica por una aproximación aprendida, reduciendo drásticamente la varianza a costa de un sesgo inicial (Trade-off Sesgo-Varianza).

---

_(Escribe tu respuesta aquí debajo)_

**[RESPUESTA DEL ESTUDIANTE]:**


## 2. Control Continuo Estable: El Colapso de A2C vs. la Robustez de PPO (12 puntos)

### Objetivo:

Evaluar el comportamiento y la vulnerabilidad de A2C frente a pasos de gradiente destructivos en espacios de acción continua, contrastándolo con las garantías de estabilidad y la restricción del objetivo subrogado recortado (_Clipped Surrogate Objective_) de Proximal Policy Optimization (PPO) en el entorno **`Pendulum-v1`**.


In [ ]:
# =====================================================================
# ARQUITECTURA PARA ESPACIOS DE ACCIÓN CONTINUA (Pendulum-v1)
# =====================================================================


class ContinuousActorCritic(nn.Module):
    """
    Red Actor-Crítico para control continuo (Apartados 2.1 y 2.2).
    - Tronco compartido de extracción de características (hidden=128, ReLU).
    - Cabeza del Actor: Media mu acotada en el rango de torque [-2.0, 2.0] mediante 2.0 * tanh.
    - Desviación estándar logarítmica aprendible log(sigma) parametrizada.
    - Cabeza del Crítico: Estimación escalar del valor V(s).
    """

    def __init__(self, state_dim, action_dim, hidden=128):
        super(ContinuousActorCritic, self).__init__()
        # TODO: Define el tronco compartido (Ejm. Linear -> ReLU -> Linear -> ReLU)
        # TODO: Define la cabeza lineal para la media del actor (mu_head)
        # TODO: Define el parámetro aprendible para log(sigma) inicializado en ceros:
        #       self.log_std = nn.Parameter(torch.zeros(action_dim))
        # TODO: Define la cabeza lineal para el valor escalar del crítico (v_head)
        # === TU CÓDIGO AQUÍ ===
        self.tronco = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden, action_dim)
        self.log_std = nn.Parameter(torch.zeros(action_dim))
        self.v_head = nn.Linear(hidden, 1)

    def forward(self, x):
        # TODO: Pasa x por el tronco compartido
        # TODO: Acota la media mu al rango de torque [-2.0, 2.0] usando 2.0 * torch.tanh(...)
        # TODO: Calcula std = self.log_std.exp().clamp(1e-3, 2.0)
        # TODO: Crea la distribución Normal(mu, std)
        # TODO: Calcula el valor escalar del crítico V(s)
        # Retorna: dist, val
        # === TU CÓDIGO AQUÍ ===
        features = self.tronco(x)
        mu = 2.0 * torch.tanh(self.mu_head(features))
        std = self.log_std.exp().clamp(1e-3, 2.0)
        dist = Normal(mu, std)
        val = self.v_head(features)
        return dist, val

In [ ]:
def train_a2c_continuous(
    env_name="Pendulum-v1", iterations=150, steps_per_iter=2048, lr=3e-4, gamma=0.99
):
    """
    Apartado 2.1: A2C en espacio continuo con actualización on-policy.

    Requisitos:
    1. Utilizar la arquitectura Gaussiana continua `ContinuousActorCritic`.
    2. Recolectar transiciones continuas muestreando acciones de la Normal.
    3. Normalizar la recompensa (Ejm. reward / 8.0) para estabilizar la función de valor.
    4. Calcular la ventaja TD de 1 paso y optimizar el modelo en cada paso.
    5. Registrar el promedio de recompensa por episodio en `ep_rewards_history`.
    """
    torch.manual_seed(42)
    np.random.seed(42)
    env = gym.make(env_name)
    env.reset(seed=42)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]

    model = ContinuousActorCritic(state_dim, action_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # IMPORTANTE: Almacena aquí la recompensa promedio de cada iteración para que
    # las celdas de visualización 2.3 y 2.4 grafiquen automáticamente.
    ep_rewards_history = []

    for it in range(iterations):
        state, _ = env.reset()
        ep_reward = 0
        episodes_completed = 0
        total_rewards = 0

        for step in range(steps_per_iter):
            state_tensor = torch.as_tensor(
                state, dtype=torch.float32, device=device
            ).unsqueeze(0)

            # TODO 2.1.A: Evaluar el estado en el modelo, muestrear acción continua de la Normal
            # y acotar al rango de acciones permitido (np.clip con env.action_space.low y high)
            # === TU CÓDIGO AQUÍ ===
            dist, val = model(state_tensor)
            action = dist.sample()
            action_np = np.clip(
                action.detach().cpu().numpy(),
                env.action_space.low,
                env.action_space.high,
            )

            next_state, reward, terminated, truncated, _ = env.step(action_np.flatten())
            done = terminated or truncated

            next_state_tensor = torch.as_tensor(
                next_state, dtype=torch.float32, device=device
            ).unsqueeze(0)
            with torch.no_grad():
                _, val_next = model(next_state_tensor)
            if done:
                val_next = torch.zeros_like(val_next)

            # TODO 2.1.B: Calcular TD Target y Ventaja TD de 1 paso continuo
            # Escala reward por 8.0 para estabilizar la escala del valor: (reward / 8.0)
            # td_target = (reward / 8.0) + gamma * val_next
            # advantage = td_target - val
            # === TU CÓDIGO AQUÍ ===
            td_target = (reward / 8.0) + gamma * val_next
            advantage = td_target - val

            # TODO 2.1.C: Calcular pérdidas del actor continuo (-dist.log_prob(action) * advantage.detach())
            # y del crítico (MSE entre val y td_target), y actualizar parámetros con optimizer
            # === TU CÓDIGO AQUÍ ===
            log_prob = dist.log_prob(action).sum(dim=-1, keepdim=True)
            actor_loss = -(log_prob * advantage.detach()).mean()
            critic_loss = F.mse_loss(val, td_target)
            loss = actor_loss + 0.5 * critic_loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            ep_reward += reward
            state = next_state

            if done:
                total_rewards += ep_reward
                episodes_completed += 1
                ep_reward = 0
                state, _ = env.reset()

        # Almacenar recompensa promedio de la iteración
        avg_reward = (
            (total_rewards / episodes_completed)
            if episodes_completed > 0
            else (ep_rewards_history[-1] if ep_rewards_history else -1500.0)
        )
        ep_rewards_history.append(avg_reward)

        if (it + 1) % 25 == 0 or it == 0:
            print(
                f"[A2C Continuo] Iteración {it + 1:3d}/{iterations} | Recompensa promedio: {avg_reward:.2f}"
            )

    env.close()
    return ep_rewards_history

In [ ]:
def train_ppo_continuous(
    env_name="Pendulum-v1",
    iterations=150,
    steps_per_iter=2048,
    lr=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    eps_clip=0.2,
    K_epochs=5,
    batch_size=64,
):
    """
    Apartado 2.2: Proximal Policy Optimization (PPO) Continuo con Clipped Surrogate Objective.

    Requisitos:
    1. Recolectar trayectorias de tamaño T = 2048 (estados, acciones, logprobs_old, values, rewards).
    2. Calcular Generalized Advantage Estimation (GAE con lambda = 0.95).
    3. Normalizar ventajas: (advantages - mean) / (std + 1e-8).
    4. Optimizar durante K = 5 épocas dividiendo los 2048 pasos en mini-batches de tamaño 64.
    5. Calcular ratio de verosimilitud: r_t(theta) = exp(logprob_new - logprob_old).
    6. Calcular el objetivo surrogate recortado:
           L_CLIP = -min(r_t * A_t, clip(r_t, 1-eps, 1+eps) * A_t)
    7. Registrar la fracción de clipping (pasos donde r_t < 1-eps o r_t > 1+eps).
    8. Retornar `ep_rewards_history` y `clip_fractions_history`.
    """
    torch.manual_seed(42)
    np.random.seed(42)
    env = gym.make(env_name)
    env.reset(seed=42)
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]

    model = ContinuousActorCritic(state_dim, action_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    # IMPORTANTE: Almacenar aquí las recompensas promedio y las tasas de clipping
    # para que las celdas de visualización 2.3 y 2.4 grafiquen automáticamente.
    ep_rewards_history = []
    clip_fractions_history = []

    for it in range(iterations):
        states, actions, logprobs, rewards, values, masks = [], [], [], [], [], []
        state, _ = env.reset()
        ep_reward = 0
        episodes_completed = 0
        total_rewards = 0

        # --- Fase 1: Recolección de Trayectorias (T = 2048) ---
        for step in range(steps_per_iter):
            state_tensor = torch.as_tensor(
                state, dtype=torch.float32, device=device
            ).unsqueeze(0)
            with torch.no_grad():
                dist, val = model(state_tensor)
                action = dist.sample()
            action_np = (
                action.cpu().numpy().clip(env.action_space.low, env.action_space.high)
            )

            next_state, reward, terminated, truncated, _ = env.step(action_np.flatten())
            done = terminated or truncated

            states.append(state)
            actions.append(action)
            logprobs.append(dist.log_prob(action).sum(dim=-1, keepdim=True))
            values.append(val.item())
            rewards.append(reward / 8.0)  # Escalar recompensa para estabilidad
            masks.append(1 - int(done))

            ep_reward += reward
            state = next_state

            if done:
                total_rewards += ep_reward
                episodes_completed += 1
                ep_reward = 0
                state, _ = env.reset()

        avg_reward = (
            (total_rewards / episodes_completed)
            if episodes_completed > 0
            else (ep_rewards_history[-1] if ep_rewards_history else -1500.0)
        )
        ep_rewards_history.append(avg_reward)

        # --- Fase 2: Cálculo de Retornos y GAE (lambda = 0.95) ---
        next_state_tensor = torch.as_tensor(
            state, dtype=torch.float32, device=device
        ).unsqueeze(0)
        with torch.no_grad():
            _, next_val = model(next_state_tensor)
        next_val = next_val.item()

        # TODO 2.2.A: Implementar el cálculo de GAE hacia atrás
        # delta = rewards[step] + gamma * next_val * masks[step] - values[step]
        # gae = delta + gamma * gae_lambda * masks[step] * gae
        # returns.insert(0, gae + values[step])
        # === TU CÓDIGO AQUÍ ===
        returns = []
        gae = 0.0
        for step in reversed(range(steps_per_iter)):
            delta = rewards[step] + gamma * next_val * masks[step] - values[step]
            gae = delta + gamma * gae_lambda * masks[step] * gae
            returns.insert(0, gae + values[step])
            next_val = values[step]

        # Preparación de tensores en memoria (Provisto)
        b_states = torch.as_tensor(np.array(states), dtype=torch.float32, device=device)
        b_actions = torch.cat(actions).view(-1, action_dim)
        b_old_logprobs = torch.cat(logprobs).view(-1, 1)
        b_returns = torch.tensor(returns, dtype=torch.float32, device=device).unsqueeze(
            1
        )
        b_values = torch.tensor(values, dtype=torch.float32, device=device).unsqueeze(1)

        # TODO 2.2.B: Calcular y normalizar ventajas
        # b_advantages = b_returns - b_values
        # Normalizar: (b_advantages - b_advantages.mean()) / (b_advantages.std() + 1e-8)
        # === TU CÓDIGO AQUÍ ===
        b_advantages = b_returns - b_values
        b_advantages = (b_advantages - b_advantages.mean()) / (
            b_advantages.std(unbiased=False) + 1e-8
        )

        # --- Fase 3: Optimización PPO (K épocas y mini-batches de 64) ---
        dataset_size = steps_per_iter
        iter_clip_fractions = []

        # TODO 2.2.C: Bucle de optimización PPO:
        # 1. Iterar por K_epochs épocas.
        # 2. Generar permutación aleatoria de índices: permutation = torch.randperm(dataset_size)
        # 3. Recorrer en bloques de tamaño batch_size (64):
        #    - Extraer mini-batches: mb_states, mb_actions, mb_old_logprobs, mb_returns, mb_advantages
        #    - Evaluar modelo: dist, val = model(mb_states)
        #    - Calcular logprobs_new = dist.log_prob(mb_actions)
        #    - Calcular ratio r_t = exp(logprobs_new - mb_old_logprobs)
        #    - Calcular surr1 = r_t * mb_advantages
        #    - Calcular surr2 = clamp(r_t, 1.0 - eps_clip, 1.0 + eps_clip) * mb_advantages
        #    - Actor Loss = -torch.min(surr1, surr2).mean()
        #    - Critic Loss = F.mse_loss(val, mb_returns)
        #    - Loss = Actor Loss + 0.5 * Critic Loss
        #    - Optimizar: optimizer.zero_grad(), loss.backward(), optimizer.step()
        #    - Registrar fracción de clipping:
        #      clipped = (r_t < 1.0 - eps_clip) | (r_t > 1.0 + eps_clip)
        #      iter_clip_fractions.append(clipped.float().mean().item())
        # === TU CÓDIGO AQUÍ ===
        for epoch in range(K_epochs):
            permutation = torch.randperm(dataset_size, device=device)
            for start in range(0, dataset_size, batch_size):
                idx = permutation[start : start + batch_size]
                mb_states, mb_actions = b_states[idx], b_actions[idx]
                mb_old_logprobs = b_old_logprobs[idx]
                mb_returns, mb_advantages = b_returns[idx], b_advantages[idx]
                dist, val = model(mb_states)
                logprobs_new = dist.log_prob(mb_actions).sum(dim=-1, keepdim=True)
                ratio = (logprobs_new - mb_old_logprobs).exp()
                surr1 = ratio * mb_advantages
                surr2 = ratio.clamp(1.0 - eps_clip, 1.0 + eps_clip) * mb_advantages
                actor_loss = -torch.min(surr1, surr2).mean()
                critic_loss = F.mse_loss(val, mb_returns)
                loss = actor_loss + 0.5 * critic_loss
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                clipped = (ratio < 1.0 - eps_clip) | (ratio > 1.0 + eps_clip)
                iter_clip_fractions.extend(clipped.detach().flatten().cpu().tolist())

        avg_clip_fraction = (
            np.mean(iter_clip_fractions) if len(iter_clip_fractions) > 0 else 0.0
        )
        clip_fractions_history.append(avg_clip_fraction)

        if (it + 1) % 25 == 0 or it == 0:
            print(
                f"[PPO Continuo (eps={eps_clip})] Iteración {it + 1:3d}/{iterations} | Recompensa: {avg_reward:.2f} | Clipping Frac: {avg_clip_fraction:.3f}"
            )

    env.close()
    return ep_rewards_history, clip_fractions_history

In [ ]:
# =====================================================================
# BLOQUE DE EJECUCIÓN Y VISUALIZACIÓN DEL EJERCICIO 2.3 (PROVISTO)
# =====================================================================

print("===============================================================")
print(" ENTRENAMIENTO EJERCICIO 2.3: Pendulum-v1 (150 Iteraciones)")
print("===============================================================")

iters_ex2 = 150

print(f"\n--- 1. Entrenando A2C Continuo ({iters_ex2} iteraciones)... ---")
rewards_a2c_cont = train_a2c_continuous(iterations=iters_ex2)

print(f"\n--- 2. Entrenando PPO Continuo (eps=0.2, {iters_ex2} iteraciones)... ---")
rewards_ppo_cont, clip_fractions_ppo = train_ppo_continuous(
    iterations=iters_ex2, eps_clip=0.2
)

# --- Gráfica 1: Recompensa Promedio (A2C vs PPO) ---
plt.figure(figsize=(12, 5))
plt.plot(
    rewards_a2c_cont,
    color="crimson",
    linewidth=1.8,
    alpha=0.85,
    label="A2C Continuo (Colapso / Estancamiento)",
)
plt.plot(
    rewards_ppo_cont,
    color="forestgreen",
    linewidth=2.2,
    label="PPO Continuo (Convergencia Estable)",
)
plt.axhline(
    y=-200,
    color="black",
    linestyle="--",
    linewidth=1.5,
    label="Zona Óptima Pendulum-v1 (>= -200)",
)
plt.title(
    "Ejercicio 2.3: Curvas de Recompensa Promedio en Pendulum-v1 (150 Iteraciones)",
    fontsize=13,
    fontweight="bold",
)
plt.xlabel("Iteraciones de Entrenamiento (cada iteración = 2048 pasos)", fontsize=11)
plt.ylabel("Recompensa Promedio por Episodio", fontsize=11)
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

# --- Gráfica 2: Variación del Clipping de PPO (Pedido explícitamente en apartado 2.3 del PDF) ---
plt.figure(figsize=(12, 4))
plt.plot(
    clip_fractions_ppo,
    color="teal",
    linewidth=2.0,
    label="Fracción de Clipping (eps=0.2)",
)
plt.title(
    "Ejercicio 2.3: Evolución de la Tasa de Clipping de PPO durante el Entrenamiento",
    fontsize=13,
    fontweight="bold",
)
plt.xlabel("Iteraciones de Entrenamiento", fontsize=11)
plt.ylabel("Fracción de Datos Recortados (%)", fontsize=11)
plt.ylim(0, 0.5)
plt.legend(loc="upper right")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# EXPERIMENTO 2.4: ANÁLISIS DE FACTOR DE CLIPPING (PPO con eps=1.0) (PROVISTO)
# =====================================================================

print("===============================================================")
print(" EXPERIMENTO 2.4: Análisis de Factor de Clipping (PPO con eps=1.0)")
print("===============================================================")

print("Entrenando PPO con eps=1.0 durante 30 iteraciones...")
rewards_ppo_unclipped, clip_fractions_unclipped = train_ppo_continuous(
    iterations=30, eps_clip=1.0, K_epochs=5, batch_size=64
)

# Comparación gráfica de las primeras 30 iteraciones
plt.figure(figsize=(12, 5))
plt.plot(
    rewards_a2c_cont[:30],
    color="crimson",
    linestyle=":",
    linewidth=2.0,
    label="A2C Continuo (Primeras 30 iters)",
)
plt.plot(
    rewards_ppo_cont[:30],
    color="forestgreen",
    linewidth=2.2,
    label="PPO Normal (eps=0.2, Clipping activo)",
)
plt.plot(
    rewards_ppo_unclipped,
    color="purple",
    linewidth=2.0,
    linestyle="--",
    label="PPO sin Clipping (eps=1.0)",
)

plt.title(
    "Ejercicio 2.4: Impacto del Factor de Clipping (Primeras 30 Iteraciones)",
    fontsize=13,
    fontweight="bold",
)
plt.xlabel("Iteraciones de Entrenamiento", fontsize=11)
plt.ylabel("Recompensa Promedio", fontsize=11)
plt.axhline(y=-200, color="gray", linestyle="--", label="Zona Óptima (-200)")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.show()

### 2.4. Análisis de Factor Clipping: Diagnóstico con $\epsilon = 1.0$ (1.5 puntos)

_Entrena durante 30 iteraciones el PPO con $\epsilon=1.0$, indica qué ocurre y si es parecido al colapso de A2C._

---

**Tu análisis debe responder:**

1. ¿A qué intervalo se expande la región de recorte $[1-\epsilon, 1+\epsilon]$ cuando $\epsilon = 1.0$? ¿Qué factor de cambio relativo permite esto en la probabilidad de una acción?
2. ¿Qué ocurre con la fracción de clipping registrada en esta prueba en comparación con $\epsilon = 0.2$?
3. Observando la curva de recompensas de las 30 iteraciones, ¿el agente logra aprender o se estanca/colapsa? ¿Es este fenómeno similar al observado en A2C continuo? Justifica por qué el clipping es el verdadero responsable de la estabilidad de PPO frente al colapso de política.

---

_(Escribe tu respuesta aquí debajo)_

**[RESPUESTA DEL ESTUDIANTE]:**


### 2.5. Pregunta de Análisis: Mecanismo de Clipping vs. Colapso de A2C (1.5 puntos)

_Observe el comportamiento del entrenamiento. Es altamente probable que note que la curva de A2C continuo colapsa o se estanca en rendimientos deficientes (cerca de $-1000$ o peores), mientras que PPO converge de forma estable hacia la zona óptima (entre $-150$ y $-250$). Explique textualmente y con rigor matemático cómo el mecanismo de clipping basado en el ratio $r_t(\theta)$ previene que PPO sufra el colapso que afecta a A2C cuando se dan pasos de gradiente destructivos en entornos continuos._

---

**Puntos clave a desarrollar formalmente:**

1. **Gradiente en Políticas Gaussianas:** Escribe la derivada de $\log \pi_\theta(a|s)$ respecto a la media $\mu$:
   $$\nabla_\mu \log \pi_\theta(a|s) = \frac{a - \mu}{\sigma^2}$$
   Explica cómo una estimación ruidosa de la ventaja combinada con $\sigma^2$ en el denominador puede provocar un salto desmesurado en $\mu$ y colapsar la política hacia acciones extremas (torque $\pm 2.0$) de forma irreversible en A2C.
2. **Formulación de PPO:** Escribe la ecuación de $L^{CLIP}(\theta)$ y define el ratio $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$.
3. **Análisis de Casos y Anulación del Gradiente:** Demuestra analíticamente por qué la función $\min(\dots)$ satura y genera un gradiente nulo ($\nabla_\theta = 0$) cuando:
   - $\hat{A}_t > 0$ y $r_t(\theta) > 1 + \epsilon$.
   - $\hat{A}_t < 0$ y $r_t(\theta) < 1 - \epsilon$.
4. **Región de Confianza:** Concluye cómo este límite pesimista actúa de forma equivalente a restringir la divergencia KL ($D_{KL}(\pi_{old} \parallel \pi) \le \delta$), impidiendo el olvido catastrófico y asegurando mejoras monótonas.

---

_(Escribe tu respuesta aquí debajo)_

**[RESPUESTA DEL ESTUDIANTE]:**
